# Environment audit

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet **ON** (notebooks 02–06 download the 8 pinned helper wheels, ~57 MB, and 05/06 download the model). Run notebooks in order 01 → 06 (each Kaggle notebook is a fresh session, so notebooks 03–06 re-run the idempotent installer first); each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Prereq: release dataset attached. Success: every check prints PASS (Python 3.13, torch 2.11.0+cu130, Triton 3.6, glibc>=2.38, 2x Tesla T4 SM75). Peer access/topology are recorded, not assumed.

## 1. Setup — locate the release bundle

In [1]:
import glob, hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor

HF_REPO = "waqasm86/stokastisk"
HF_BASE = f"https://huggingface.co/{HF_REPO}/resolve"


def _open(url, headers=None):
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers or {"User-Agent": "stokastisk-fetch/1"}), timeout=60)


def _range_ok(url):
    if not url.startswith("http"):
        return None
    with _open(url, {"Range": "bytes=0-0", "User-Agent": "stokastisk-fetch/1"}) as r:
        cr = r.headers.get("Content-Range")                      # "bytes 0-0/37080766"
        return int(cr.rsplit("/", 1)[1]) if r.status == 206 and cr else None


def _download(url, dest, workers=1, retries=3):
    """Download to `<dest>.part`, check free disk space and the advertised length, then replace `dest` atomically; the partial file is removed on any failure."""
    dest = pathlib.Path(dest); part = dest.with_name(dest.name + ".part")
    part.unlink(missing_ok=True)
    try:
        total = _range_ok(url)
        if total and shutil.disk_usage(dest.parent).free < total + (64 << 20):
            raise RuntimeError(f"not enough free disk space in {dest.parent} for {total / 1e6:.0f} MB")
        _download_to(url, part, workers, retries)
        if total and part.stat().st_size != total:
            raise RuntimeError(f"incomplete download: {part.stat().st_size} of {total} bytes")
        os.replace(part, dest)
    except BaseException:
        part.unlink(missing_ok=True)
        raise


def _download_to(url, dest, workers=1, retries=3):
    dest = pathlib.Path(dest); total = _range_ok(url)
    t0 = time.time()
    if total and total > (4 << 20):
        chunk = -(-total // workers); parts = [(i * chunk, min(total, (i + 1) * chunk) - 1) for i in range(workers) if i * chunk < total]
        with open(dest, "wb") as f:
            f.truncate(total)

        def part(rng):
            a, b = rng
            for attempt in range(retries):
                try:
                    with _open(url, {"Range": f"bytes={a}-{b}", "User-Agent": "stokastisk-fetch/1"}) as r, open(dest, "r+b") as f:
                        f.seek(a); shutil.copyfileobj(r, f, 1 << 20)
                    return
                except Exception:
                    if attempt == retries - 1: raise
                    time.sleep(2 ** attempt)
        with ThreadPoolExecutor(len(parts)) as ex:
            list(ex.map(part, parts))
        mode = f"{len(parts)} parallel ranges"
    else:
        for attempt in range(retries):
            try:
                with _open(url) as r, open(dest, "wb") as f:
                    shutil.copyfileobj(r, f, 1 << 20)
                break
            except Exception:
                if attempt == retries - 1: raise
                time.sleep(2 ** attempt)
        mode = "single stream"
    n = dest.stat().st_size
    print(f"downloaded {n / 1e6:.1f} MB in {time.time() - t0:.1f}s ({n / 1e6 / max(time.time() - t0, 1e-6):.1f} MB/s, {mode})")


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def fetch_release(release, dest="/tmp/stokastisk-release", base=None, revision="main", workers=1):
    """Return the verified, extracted bundle directory (…/<release>). `base` overrides the Hub URL prefix (tests use file://)."""
    base = (base or HF_BASE).rstrip("/")
    root = f"{base}/{revision}/releases/{release}" if base.startswith("http") else f"{base}/releases/{release}"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    zname = f"{release}-kaggle-input.zip"; zpath = dest / zname
    if zpath.exists(): zpath.unlink()
    with _open(f"{root}/{zname}.sha256") as r:
        expected = r.read().decode().split()[0]
    _download(f"{root}/{zname}", zpath, workers=workers)
    got = _sha256(zpath)
    if got != expected:
        zpath.unlink(); raise RuntimeError(f"SHA256 mismatch for {zname}: expected {expected}, got {got}")
    out = dest / release
    if out.exists(): shutil.rmtree(out)
    with zipfile.ZipFile(zpath) as z:
        assert z.testzip() is None
        for n in z.namelist():
            if n.startswith("/") or ".." in pathlib.PurePosixPath(n).parts or not n.startswith(release + "/"):
                raise RuntimeError(f"unsafe or foreign path in archive: {n}")
        z.extractall(dest)
    m = json.loads((out / "manifests" / "RELEASE-MANIFEST.json").read_text())
    if m.get("name") != release:
        raise RuntimeError(f"manifest name {m.get('name')!r} != {release!r}")
    print(f"release {release}: SHA256 {got[:16]}... verified, extracted to {out}")
    return out

# Explicit release identity: only the bundle directory named RELEASE whose RELEASE-MANIFEST.json carries the same name is accepted
# (an older rc is never silently selected). Source 1: the attached Kaggle dataset. Source 2 (nothing attached): the Hugging Face CDN.
RELEASE = "stokastisk-0.1.0-kaggle-t4-sm75"
cands = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
if os.environ.get("STOKASTISK_BUNDLE"): cands = [pathlib.Path(os.environ["STOKASTISK_BUNDLE"])]
match = [c for c in cands if c.name == RELEASE]
if not match:
    print(f"{RELEASE} is not attached (attached: {sorted(str(c) for c in cands)}); downloading it from Hugging Face (needs Internet ON) ...")
    match = [fetch_release(RELEASE)]
assert len({(c / "manifests" / "SHA256SUMS").read_bytes() for c in match}) == 1, f"several DIFFERENT copies of {RELEASE}: {match}"
BUNDLE = match[0]
_m = json.loads((BUNDLE / "manifests" / "RELEASE-MANIFEST.json").read_text())
assert _m.get("name") == RELEASE, f"manifest name {_m.get('name')!r} != expected {RELEASE!r}"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("release:", RELEASE, "| bundle:", BUNDLE)

release: stokastisk-0.1.0-kaggle-t4-sm75 | bundle: /kaggle/input/datasets/waqasm86/stokastisk-0-1-0-kaggle-t4-sm75-kaggle-input-2/stokastisk-0.1.0-kaggle-t4-sm75


## 2. Run the audit (read-only: installs and changes nothing)

In [2]:
import subprocess, sys, os
def run_stage(script, *args):
    """Run a bundled stage script; returns its exit code (a failure is asserted in the final verdict cell, after evidence is packaged)."""
    r = subprocess.run([sys.executable, str(SCRIPTS / script), *args], cwd=SCRIPTS,
                       env=dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", "")))
    print(f"{script} exit code: {r.returncode}")
    return r.returncode

In [3]:
RC = {"01_env_audit": run_stage("01_env_audit.py")}

PASS python 3.13 -- 3.13.16
PASS torch 2.11.0+cu130 -- 2.11.0+cu130 cuda=13.0
PASS triton 3.6.0 -- 3.6.0
PASS glibc >= 2.38 (native extensions need 2.38) -- 2.39
PASS CUDA driver initialises (cuInit(0)==0)
PASS two GPUs visible to PyTorch -- 2
PASS gpu0 Tesla T4 / SM75 -- Tesla T4 7.5
PASS gpu1 Tesla T4 / SM75 -- Tesla T4 7.5
PASS NVIDIA driver and GPU UUIDs reported by nvidia-smi -- 0, GPU-4e919dea-3890-4edf-de35-d98cc2e24e62, Tesla T4, 580.178.04, 15360 MiB
1, GPU-50cd2d9c-2300-049c-3656-f221bfd97270, Tesla T4, 580.178.04, 15360 MiB
PASS NCCL available to PyTorch -- 2.28.9
PASS baseline has no FlashAttention/FlashInfer -- none
PASS 4 CPU cores -- 4
PASS ~30 GiB system RAM -- 31.3
PASS ~15 GiB VRAM per GPU -- [14.56, 14.56]
{
 "timestamp_utc": "2026-11-11T14:47:34.245933+00:00",
 "python": "3.13.16",
 "torch": "2.11.0+cu130",
 "torch_cuda": "13.0",
 "cuda_runtime_torch": "13.0",
 "triton": "3.6.0",
 "nccl": "2.28.9",
 "nvcc": "Cuda compilation tools, release 13.0, V13.0.88",
 "glibc":

## Package the evidence

Creates `/kaggle/working/01-stokastisk-acceptance.zip` (download it from the Output tab).

In [4]:
import shutil
print(shutil.make_archive("/kaggle/working/01-stokastisk-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))

/kaggle/working/01-stokastisk-acceptance.zip


## Verdict — this cell fails the notebook if any stage failed

In [5]:
rcs = RC
assert all(v == 0 for v in rcs.values()), f'FAILED stage(s): {rcs} -- see the logs and JSON in /kaggle/working/stokastisk-acceptance/'
print('ALL STAGES PASSED', rcs)

ALL STAGES PASSED {'01_env_audit': 0}
